**SHAPING LAYERS**

In [11]:
import torch
import torch.nn as nn

# Simulate a batch of 1 image, with 1 color channel, 4x4 pixels
# Shape: (Batch, Channels, Height, Width)
dummy_image = torch.tensor([[[
    [1.0, 2.0,  0.5, 1.2],
    [3.0, 4.0,  1.1, 0.2],
    [0.1, 0.0,  5.0, 2.0],
    [1.0, 1.2,  3.0, 4.0]
]]])

# Max Pooling (2x2 window, sliding by 2 pixels at a time)
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
out_max = max_pool(dummy_image)

print(f"Original Shape: {dummy_image.shape}")
print(f"Max Pool Shape: {out_max.shape}")
print("Max Pool Output:\n", out_max[0, 0])

Original Shape: torch.Size([1, 1, 4, 4])
Max Pool Shape: torch.Size([1, 1, 2, 2])
Max Pool Output:
 tensor([[4.0000, 1.2000],
        [1.2000, 5.0000]])


In [12]:
# 1. Simple Interpolation (Upsampling by 2x)
upsample = nn.Upsample(scale_factor=2, mode='nearest')
out_up = upsample(out_max) # Taking the 2x2 from above and making it 4x4
print(f"Upsample Shape: {out_up.shape}") # (1, 1, 4, 4)

# 2. Learnable Upsampling (Transposed Convolution)
# Expands a 2x2 grid into a 4x4 grid using learned weights
conv_transpose = nn.ConvTranspose2d(
    in_channels=1,
    out_channels=1,
    kernel_size=2,
    stride=2
)
out_learned = conv_transpose(out_max)
print(f"ConvTranspose Shape: {out_learned.shape}") # (1, 1, 4, 4)

Upsample Shape: torch.Size([1, 1, 4, 4])
ConvTranspose Shape: torch.Size([1, 1, 4, 4])


In [13]:
class ConvToDenseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # nn.Flatten collapses dimensions 1 through the end into a single dimension
        self.flatten = nn.Flatten()
        # We must manually calculate the input size for the linear layer!
        # If input is 32x32 image:
        # After Pool: 16x16 image with 16 channels = 16 * 16 * 16 = 4096
        self.fc = nn.Linear(4096, 10)

    def forward(self, x):
        # x shape: (Batch, 3, 32, 32)
        x = self.conv(x)
        x = self.pool(x)       # Shape: (Batch, 16, 16, 16)
        x = self.flatten(x)    # Shape: (Batch, 4096)
        x = self.fc(x)         # Shape: (Batch, 10)
        return x

# Alternatively, using PyTorch's .view() method dynamically:
dummy_tensor = torch.randn(8, 16, 5, 5) # Batch of 8
# -1 tells PyTorch: "Keep the batch size 8, and calculate the rest automatically"
reshaped_tensor = dummy_tensor.view(8, -1)
print(f"Dynamic View Shape: {reshaped_tensor.shape}") # (8, 400)

Dynamic View Shape: torch.Size([8, 400])
